Data pipeline for analyzing and visualizing quantitative fluorescence data mapped onto the Allen Brain Atlas ontology.

Annotations and Detections CSVs are generated from ABBA alignment and QuPath puncta detection analysis

1.  **Data Processing**: Load and cache raw data from `Annotations.csv` and `Detections.csv` as feather files for faster future loading. Assign the measurements of each detection to each region to get puncta area and intensity.
2.  **Ontology**: Use the `allensdk` to map the region acronyms found in the `Name` and `Region` column to the official Allen Brain Atlas ontology, including the full anatomical name (`allen_full_name`), a unique integer ID (`allen_id`), and the parent's ID (`allen_parent_id`).
3.  **Export**: Imports a list of regions and exports measurement for those regions.

# Loading and Caching data

## Configuration

In [ ]:
import pandas as pd
import os
from pathlib import Path
import time

In [ ]:
# --- Configuration ---
# Directory containing source CSV files.
DATA_DIR = Path("./data")

# Don't include the extension
# For your two primary data types: annotations and detections. Assumes both are in DATA_DIR
ANNOTATIONS_BASE_NAME = "Annotations"
DETECTIONS_BASE_NAME = "Detections"

# --- You usually don't have to modify anything past this---
# Outputs
# Subfolder for Feather files in this notebook's directory.
FEATHER_OUTPUT_FOLDER = "Intermediate Files"
# Subfolder for exported aggregated data.
EXPORT_DIRECTORY = "Exports"
EXPORT_FILENAME_PREFIX = f"{DETECTIONS_BASE_NAME}Aggregated" # output is EXPORT_FILENAME_PREFIX_YYYYMMDD-HHMMSS

# --- Dynamically Generated Filenames and Folders ---
# Filenames are derived by adding the .csv or .feather suffix to the base names.
current_working_dir = Path.cwd()
feather_dir_path = current_working_dir / FEATHER_OUTPUT_FOLDER
feather_dir_path.mkdir(parents=True, exist_ok=True)
annotations_csv_path = DATA_DIR / Path(ANNOTATIONS_BASE_NAME).with_suffix(".csv").name
detections_csv_path = DATA_DIR / Path(DETECTIONS_BASE_NAME).with_suffix(".csv").name
annotations_feather_path = feather_dir_path / Path(ANNOTATIONS_BASE_NAME).with_suffix(".feather").name
detections_feather_path = feather_dir_path / Path(DETECTIONS_BASE_NAME).with_suffix(".feather").name
cache_ledger_name = feather_dir_path / "ledger.json"
# --- Print for Verification ---
print(f"--- Created folders ---")
print(f"  Feather Output Folder: {feather_dir_path}")
print(f"  Aggregated Export Directory: {EXPORT_DIRECTORY}")

## Functions

In [ ]:
import hashlib
import json
import pandas as pd
import time
from pathlib import Path

def _calculate_file_hash(file_path: Path) -> str:
    """Calculates the SHA256 hash of a file to verify its contents."""
    hasher = hashlib.sha256()
    try:
        with open(file_path, 'rb') as f:
            for block in iter(lambda: f.read(65536), b''):
                hasher.update(block)
        return hasher.hexdigest()
    except FileNotFoundError:
        print(f"  Warning: File not found for hashing: '{file_path.name}'")
    except Exception as e:
        print(f"  Error hashing '{file_path.name}': {e}")
    return ""

def _load_filename_mapping() -> dict:
    """Loads the cache ledger from a JSON file."""
    if not cache_ledger_name.exists():
        return {}
    try:
        with open(cache_ledger_name, 'r') as f:
            return json.load(f)
    except json.JSONDecodeError:
        print(f"  Warning: Invalid cache ledger. Starting fresh.")
        return {}

def _save_filename_mapping(mapping: dict) -> None:
    """Saves the cache ledger to a JSON file."""
    try:
        with open(cache_ledger_name, 'w') as f:
            json.dump(mapping, f, indent=4)
    except Exception as e:
        print(f"  Error saving cache ledger: {e}")

def _generate_unique_feather_path(csv_path: Path) -> Path:
    """
    Generates a unique, human-readable path for a new Feather file.
    e.g., 'parent_dir_data_0814_123500.feather'
    """
    base_name = csv_path.stem
    parent_name = csv_path.parent.name.replace(" ", "_").replace(".", "_")
    timestamp = time.strftime("%m%d_%H%M%S")

    candidate_stem = f"{parent_name}_{base_name}_{timestamp}"
    candidate_path = feather_dir_path / f"{candidate_stem}.feather"

    # Handle the extremely rare case of a filename collision
    counter = 1
    final_path = candidate_path
    while final_path.exists():
        final_path = feather_dir_path / f"{candidate_stem}_{counter}.feather"
        counter += 1

    return final_path

def _load_csv(csv_path: Path) -> pd.DataFrame | None:
    """Loads a CSV file into a DataFrame, with basic error handling."""
    print(f"  Loading CSV: '{csv_path.name}'...")
    try:
        start_time = time.time()
        df = pd.read_csv(csv_path, low_memory=False)
        print(f"  -> CSV loaded in {time.time() - start_time:.2f}s. Shape: {df.shape}")
        return df
    except Exception as e:
        print(f"  -> Error loading CSV '{csv_path}': {e}")
        return None

def _save_feather(df: pd.DataFrame, feather_path: Path) -> None:
    """Saves a DataFrame to a Feather file."""
    print(f"  Saving to Feather: '{feather_path.name}'...")
    try:
        start_time = time.time()
        df.to_feather(feather_path)
        print(f"  -> Feather saved in {time.time() - start_time:.2f}s.")
    except Exception as e:
        print(f"  -> Error saving to Feather '{feather_path}': {e}")

def _load_feather(feather_path: Path) -> pd.DataFrame | None:
    """Loads a Feather file into a DataFrame."""
    print(f"  Loading from Feather cache: '{feather_path.name}'...")
    try:
        start_time = time.time()
        df = pd.read_feather(feather_path)
        print(f"  -> Feather loaded in {time.time() - start_time:.2f}s. Shape: {df.shape}")
        return df
    except Exception as e:
        print(f"  -> Error loading Feather '{feather_path}': {e}")
        return None

def load_or_create_dataframe(csv_path_str: str) -> pd.DataFrame | None:
    """
    Loads a DataFrame from a content-addressable cache or creates it from a CSV.
    This function uses the CSV's content hash as the key to find cached data.
    """
    csv_path = Path(csv_path_str)
    print(f"\nProcessing: {csv_path.name}")

    # 1. Calculate current file hash. This is our primary key.
    current_hash = _calculate_file_hash(csv_path)
    if not current_hash:
        return None # Skip if hashing failed.

    # 2. Load cache ledger and check for a valid cache hit using the hash.
    mapping = _load_filename_mapping()
    cached_info = mapping.get(current_hash)

    if cached_info:
        feather_path = Path(cached_info['feather_path'])
        if feather_path.exists():
            print("  Cache hit. Loading directly from Feather file.")
            return _load_feather(feather_path)
        else:
            print("  Cache invalid (Feather file deleted). Re-processing.")

    # 3. Cache miss or invalid: Load from CSV.
    print("  Cache miss. Processing from CSV.")
    df = _load_csv(csv_path)
    if df is None:
        return None # Stop if CSV loading failed.

    # 4. Find a unique path and save the DataFrame as a Feather file.
    feather_path = _generate_unique_feather_path(csv_path)
    _save_feather(df, feather_path)

    # 5. Update the mapping with the new hash and save it to disk.
    mapping[current_hash] = {
        'feather_path': str(feather_path.resolve())
    }
    _save_filename_mapping(mapping)
    print(f"  -> Successfully created cache for '{csv_path.name}' with hash '{current_hash[:8]}...'.")

    return df

## Execution

In [ ]:
# --- Load DataFrames ---
df_annotations = load_or_create_dataframe(annotations_csv_path)
df_detections = load_or_create_dataframe(detections_csv_path)

# Allen Ontology (get full names)

In [ ]:
# --- Allen SDK Initialization and Ontology Mapping ---
# This section enriches the data by mapping abbreviated brain region names to their
# full anatomical names provided by the Allen Brain Atlas ontology. This makes
# subsequent analysis and visualizations more descriptive.

from allensdk.core.mouse_connectivity_cache import MouseConnectivityCache
import time

# --- 1. Setup Allen SDK and Fetch Ontology ---
# Define the output directory for Allen SDK files.
ALLEN_SDK_OUTPUT_FOLDER = "Allen SDK files"
allen_sdk_dir_path = Path.cwd() / ALLEN_SDK_OUTPUT_FOLDER
allen_sdk_dir_path.mkdir(exist_ok=True)
manifest_path = allen_sdk_dir_path / "mouse_connectivity_manifest.json"
print(f"Allen SDK cache directory set to: {allen_sdk_dir_path}")

# Initialize the cache for Allen SDK.
mcc = MouseConnectivityCache(manifest_file=str(manifest_path))

# Get the brain structure tree to build a name mapping.
try:
    structure_tree = mcc.get_structure_tree()
    structures = structure_tree.nodes()
    print("Successfully loaded Allen Brain Atlas structure tree.")

    # Create a mapping dictionary from region acronym to full name.
    acronym_map = {
        region['acronym']: region['name']
        for region in structures if 'acronym' in region and 'name' in region
    }
    print(f"   Created mapping for {len(acronym_map)} regions.")

except Exception as e:
    print(f"❌ An error occurred while fetching the Allen ontology: {e}")
    acronym_map = {} # Ensure acronym_map exists to prevent downstream errors.

# --- 2. Enrich df_annotations ---
# Replaces acronyms in the 'Name' column with their full anatomical names.
if acronym_map and 'df_annotations' in locals() and 'Name' in df_annotations.columns:
    print("\n--- Enriching 'df_annotations' (in memory) ---")
    if 'Original_Region_Name' not in df_annotations.columns:
        df_annotations['Original_Region_Name'] = df_annotations['Name']
    df_annotations['Name'] = (
        df_annotations['Original_Region_Name']
        .map(acronym_map)
        .fillna(df_annotations['Original_Region_Name'])
    )
    print("   Replaced acronyms in 'Name' column.")
    print("\nEnriched df_annotations (Head):")
    print(df_annotations[['Name', 'Original_Region_Name']].head())
else:
    print("\nSkipping 'df_annotations' enrichment (data not loaded or 'Name' column missing).")

# --- 3. Enrich df_detections ---
# Defines a helper to handle the semicolon-delimited 'Overlapping Regions' column.
def _map_overlapping_regions(region_str: str, mapping_dict: dict) -> str:
    """Maps a semicolon-delimited string of region acronyms to full names."""
    if not isinstance(region_str, str):
        return region_str

    # Gracefully handles unmapped acronyms by keeping the original acronym.
    split_regions = [r.strip() for r in region_str.split(';')]
    mapped_regions = [mapping_dict.get(r, r) for r in split_regions]
    return ';'.join(mapped_regions)

# Replaces acronyms in the 'Overlapping Regions' column.
if acronym_map and 'df_detections' in locals() and 'Overlapping Regions' in df_detections.columns:
    print("\n--- Enriching 'df_detections' (in memory) ---")
    if 'Original_Overlapping_Regions' not in df_detections.columns:
        df_detections['Original_Overlapping_Regions'] = df_detections['Overlapping Regions']

    start_time = time.time()
    df_detections['Overlapping Regions'] = df_detections['Original_Overlapping_Regions'].apply(
        lambda x: _map_overlapping_regions(x, acronym_map)
    )
    print(f"   Replaced acronyms in 'Overlapping Regions' column in {time.time() - start_time:.2f}s.")

    print("\nEnriched df_detections (Head):")
    print(df_detections[['Overlapping Regions', 'Original_Overlapping_Regions']].head())
else:
    print("\nSkipping 'df_detections' enrichment (data not loaded or 'Overlapping Regions' column missing).")

## Region Parenting and Hierarchy Visualization
To understand the relationships between brain regions and decide on the appropriate level of granularity for reporting, we can visualize the Allen Brain Atlas ontology as an interactive tree. This allows us to see how fine-grained structures (like cortical layers) roll up into their parent regions.

The following cell will generate this interactive hierarchy. You can click the arrows to expand or collapse branches to explore the structure.

In [ ]:
import pandas as pd
import json
from IPython.display import display, HTML

# --- 1. Get Structures and Create DataFrame ---
df_hierarchy = pd.DataFrame(structures)

# --- 2. Manually Create the 'parent_structure_id' Column ---
# The parent ID is the second-to-last element in the 'structure_id_path' list.
# We create this column explicitly.
df_hierarchy['parent_structure_id'] = df_hierarchy['structure_id_path'].apply(
    lambda path: path[-2] if isinstance(path, list) and len(path) > 1 else None
)

# Now select the columns we need and set the index.
df_hierarchy = df_hierarchy[['id', 'parent_structure_id', 'name', 'acronym']].copy()
df_hierarchy.set_index('id', inplace=True)

# --- 3. Build the Hierarchy as a Nested Dictionary (JSON) ---
# This recursive function converts the flat DataFrame into a nested structure suitable for a tree view.
def build_tree(df, parent_id=None):
    tree = []
    children = df[df['parent_structure_id'] == parent_id]
    if parent_id is None: # Handle the root case where parent_id is NaN/None
         children = df[df['parent_structure_id'].isna()]

    for region_id, region_data in children.iterrows():
        node = {
            'text': f"{region_data['name']} (id: {region_id})", # Add ID to label for clarity
            'id': str(region_id),
            'children': build_tree(df, region_id) # Recursively find children for this node
        }
        tree.append(node)
    return tree

print("Building the nested JSON structure for the brain hierarchy...")
# We start building from the root node, which has no parent (parent_structure_id is None).
hierarchy_json = build_tree(df_hierarchy, parent_id=None)
hierarchy_json_str = json.dumps(hierarchy_json)
print("Hierarchy built successfully.")

# --- 4. Generate and Display the Interactive HTML Tree ---
# This HTML/JS code creates the interactive tree view within the notebook output.
# (The HTML/JS part is unchanged)
html_template = f"""
<style>
    .tree-container {{
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
        font-size: 14px;
        border: 1px solid #ddd;
        border-radius: 5px;
        padding: 15px;
        overflow-x: auto; /* Add scroll for very wide content */
        max-height: 600px; /* Set a max height for scrolling */
    }}
    .tree-node ul {{
        list-style-type: none;
        padding-left: 20px;
        margin-left: 10px;
        border-left: 1px dashed #ccc;
    }}
    .tree-node li {{
        margin: 5px 0;
        position: relative;
    }}
    .tree-node .toggle {{
        cursor: pointer;
        margin-right: 5px;
        display: inline-block;
        width: 15px;
        text-align: center;
    }}
    .tree-node .toggle.collapsed::before {{
        content: '►'; /* Collapsed state */
        color: #337ab7;
    }}
    .tree-node .toggle.expanded::before {{
        content: '▼'; /* Expanded state */
        color: #337ab7;
    }}
    .tree-node .label {{
        cursor: default;
    }}
</style>

<div id="tree-root" class="tree-container"></div>

<script>
    (function() {{
        const data = {hierarchy_json_str};

        function createTree(container, nodes) {{
            const ul = document.createElement('ul');
            nodes.forEach(nodeData => {{
                const li = document.createElement('li');
                li.classList.add('tree-node');

                const toggle = document.createElement('span');
                toggle.classList.add('toggle');

                const label = document.createElement('span');
                label.classList.add('label');
                label.textContent = nodeData.text;
                li.appendChild(toggle);
                li.appendChild(label);

                if (nodeData.children && nodeData.children.length > 0) {{
                    toggle.classList.add('collapsed');
                    const childrenUl = createTree(li, nodeData.children);
                    childrenUl.style.display = 'none'; // Initially collapsed
                    li.appendChild(childrenUl);

                    toggle.onclick = function(e) {{
                        e.stopPropagation();
                        const isCollapsed = childrenUl.style.display === 'none';
                        childrenUl.style.display = isCollapsed ? 'block' : 'none';
                        toggle.classList.toggle('collapsed', !isCollapsed);
                        toggle.classList.toggle('expanded', isCollapsed);
                    }};
                }}
                ul.appendChild(li);
            }});
            container.appendChild(ul);
            return ul;
        }}

        const rootContainer = document.getElementById('tree-root');
        createTree(rootContainer, data);
        // Automatically expand the first level to show the main brain divisions
        const firstLevelToggles = rootContainer.querySelectorAll(':scope > ul > li > .toggle');
        firstLevelToggles.forEach(toggle => {{
            if (toggle.classList.contains('collapsed')) {{
                toggle.click();
            }}
        }});
    }})();
</script>
"""

display(HTML(html_template))

# Create Measurements

### Calculations

In [ ]:
# --- Configuration ---
# Define column names for input data
PUNCTA_AREA_COL = 'Subcellular cluster: Channel 2: Area'
MEAN_INTENSITY_COL = 'Subcellular cluster: Channel 2: Mean channel intensity'
NUM_SPOTS_COL = 'Num spots'
OVERLAPPING_REGIONS_COL = 'Overlapping Regions'

# Define new column names to be created
SUM_INTENSITY_COL = 'Subcellular cluster: Channel 2: Sum Intensity'
# QuPath's per-detection estimate (already area / expected-spot-area from the QuPath script)
ESTIMATED_TOTAL_SPOTS_COL = 'Num Estimated Total Spots'  # aggregated name for NUM_SPOTS_COL
REGION_COL = 'Region'                                   # Individual region names in exploded DF
AGG_SUM_INTENSITY_COL = 'Total Sum Intensity'           # Renamed aggregated SUM_INTENSITY_COL
AGG_PUNCTA_AREA_COL = 'Total Aggregated Puncta Area'    # Renamed aggregated PUNCTA_AREA_COL

print(f"--- Starting Data Processing ---")

# --- Get the content hash of the detections CSV to find its artifacts in the cache ---
detections_hash = _calculate_file_hash(detections_csv_path)
if not detections_hash:
    raise RuntimeError("Could not calculate hash for detections CSV, cannot proceed.")

print(f"  New columns to be added/calculated: '{SUM_INTENSITY_COL}' (plus regional sum of QuPath '{NUM_SPOTS_COL}' as '{ESTIMATED_TOTAL_SPOTS_COL}')")

# --- 1. Calculate Sum Intensity per Detection ---
# Convert relevant columns to numeric, coercing errors to NaN for robustness.
df_detections[PUNCTA_AREA_COL] = pd.to_numeric(df_detections[PUNCTA_AREA_COL], errors='coerce')
df_detections[MEAN_INTENSITY_COL] = pd.to_numeric(df_detections[MEAN_INTENSITY_COL], errors='coerce')

# Multiply Area by Mean Intensity to get Sum Intensity per detection.
df_detections[SUM_INTENSITY_COL] = df_detections[PUNCTA_AREA_COL] * df_detections[MEAN_INTENSITY_COL]
print(f"  Column '{SUM_INTENSITY_COL}' created in df_detections.")

# --- 2. Load or Create Exploded DataFrame with Calculated Metrics ---
df_exploded = None
# Load the cache ledger to check for an existing exploded dataframe linked to the detections CSV hash
mapping = _load_filename_mapping()
detections_cache_info = mapping.get(detections_hash, {})

# Check if a path for the exploded cache exists in the ledger
exploded_feather_path_str = detections_cache_info.get('exploded_feather_path')
if exploded_feather_path_str:
    exploded_feather_path = Path(exploded_feather_path_str)
    if exploded_feather_path.exists():
        print(f"\n  Cache hit for exploded DataFrame. Loading from '{exploded_feather_path.name}'...")
        df_exploded = _load_feather(exploded_feather_path)
    else:
        print("  Cache invalid (Exploded Feather file deleted). Re-generating.")

if df_exploded is None: # If not loaded from Feather or loading failed
    print("\n  Generating and caching exploded DataFrame from df_detections...")

    # (Calculations for CALC_NUM_CLUSTERS, etc. remain the same)
    df_detections[OVERLAPPING_REGIONS_COL] = df_detections[OVERLAPPING_REGIONS_COL].astype(str).fillna('')
    df_detections[NUM_SPOTS_COL] = pd.to_numeric(df_detections[NUM_SPOTS_COL], errors='coerce')
    df_detections[PUNCTA_AREA_COL] = pd.to_numeric(df_detections[PUNCTA_AREA_COL], errors='coerce')
    
    # Explode the 'Overlapping Regions' column into individual rows.
    df_exploded = df_detections.assign(
        **{REGION_COL: df_detections[OVERLAPPING_REGIONS_COL].str.split(';')}
    ).explode(REGION_COL)

    # Clean up region names by stripping whitespace and filtering out empty strings.
    df_exploded[REGION_COL] = df_exploded[REGION_COL].str.strip()
    df_exploded = df_exploded[df_exploded[REGION_COL] != '']
    df_exploded = df_exploded.reset_index(drop=True)

    # --- Generate a unique name and save the exploded dataframe ---
    parent_name = detections_csv_path.parent.name.replace(" ", "_").replace(".", "_")
    exploded_filename = f"{parent_name}_{DETECTIONS_BASE_NAME}_exploded_{detections_hash[:8]}.feather"
    exploded_feather_path = feather_dir_path / exploded_filename

    _save_feather(df_exploded, exploded_feather_path)

    # --- Update the ledger with the path to the new exploded file ---
    if detections_hash not in mapping:
        # This case handles a rare edge condition where the primary feather file was created
        # but the program crashed before this cell ran.
        mapping[detections_hash] = {}
    mapping[detections_hash]['exploded_feather_path'] = str(exploded_feather_path.resolve())
    _save_filename_mapping(mapping)
    print(f"  Exploded DataFrame generated and its path has been saved to the ledger.")

# --- 3. Aggregate Metrics per Region ---
print("\n  Grouping by region and summing intensity and new metrics...")

# Define the columns to be summed during aggregation.
columns_to_sum_in_agg = [
    PUNCTA_AREA_COL,
    SUM_INTENSITY_COL,
    NUM_SPOTS_COL,
]

# Group by the cleaned region names and sum the specified columns.
sum_intensity_per_region = df_exploded.groupby(REGION_COL)[columns_to_sum_in_agg].sum().reset_index()

# Rename the aggregated sum intensity column for clarity.
sum_intensity_per_region = sum_intensity_per_region.rename(
    columns={
        PUNCTA_AREA_COL: AGG_PUNCTA_AREA_COL,
        SUM_INTENSITY_COL: AGG_SUM_INTENSITY_COL,
        NUM_SPOTS_COL: ESTIMATED_TOTAL_SPOTS_COL,
    }
)

print("\n--- Aggregation Complete ---")
print(f"  Total number of unique regions found: {len(sum_intensity_per_region)}")
print("\n  Aggregated Data per Region (first 10 rows):")
sum_intensity_per_region.head()

In [ ]:
# Define the column name for Area from df_annotations
AREA_UM2_COL = 'Area µm^2'
ANNOTATION_REGION_NAME_COL = 'Name'
# New columns
AGG_AREA_UM2_COL = 'Anatomical Region Area µm^2'    # Renamed aggregated AREA_UM2_COL
AGG_AREA_PERCENT_COL = 'Total Puncta Area (%)'
CALC_NORM_INTENSITY_COL = 'Normalized Puncta Intensity (per µm^2)' # Calculated via AGG_SUM_INTENSITY_COL / AGG_AREA_UM2_COL

print("\n--- Calculating Puncta Area Percentage per Region's Anatomical Area ---")

# 1. Aggregate df_annotations to get the total anatomical area per region name
# Clean 'Name' column in df_annotations like OVERLAPPING_REGIONS_COL in df_detections.
df_annotations[ANNOTATION_REGION_NAME_COL] = df_annotations[ANNOTATION_REGION_NAME_COL].astype(str).str.strip()
# Group by the cleaned region names and sum the anatomical area.
anatomical_area_per_region = df_annotations.groupby(ANNOTATION_REGION_NAME_COL)[AREA_UM2_COL].sum().reset_index()
# Rename columns for clarity and to facilitate merging.
anatomical_area_per_region = anatomical_area_per_region.rename(columns={
    ANNOTATION_REGION_NAME_COL: REGION_COL, # Rename ANNOTATION_REGION_NAME_COL to REGION_COL to merge them easily
    AREA_UM2_COL: AGG_AREA_UM2_COL
})
name_to_id_map = df_hierarchy.reset_index().set_index('name')['id'].to_dict()
anatomical_area_per_region['id'] = anatomical_area_per_region[REGION_COL].map(name_to_id_map)

# 2. Merge sum_intensity_per_region with the anatomical_area_per_region
# This joins the puncta aggregation data with the anatomical area data for each region based on the common 'Region' column.
sum_intensity_per_region_with_anatomical_area = pd.merge(
    sum_intensity_per_region,
    anatomical_area_per_region,
    on=REGION_COL,
    how='left'
)

# 3. Calculate the percentage: (Total Aggregated Puncta Area / Region's Anatomical Area) * 100
# Initialize the new percentage column with 0.0.
sum_intensity_per_region_with_anatomical_area[AGG_AREA_PERCENT_COL] = 0.0
# Define a mask for valid anatomical area values (not NaN and not zero) to prevent errors.
valid_area_mask = (sum_intensity_per_region_with_anatomical_area[AGG_AREA_UM2_COL].notna()) & \
                  (sum_intensity_per_region_with_anatomical_area[AGG_AREA_UM2_COL] != 0)
# Apply the percentage calculation only to rows with valid anatomical areas.
sum_intensity_per_region_with_anatomical_area.loc[valid_area_mask, AGG_AREA_PERCENT_COL] = \
    (sum_intensity_per_region_with_anatomical_area[AGG_PUNCTA_AREA_COL] /
     sum_intensity_per_region_with_anatomical_area[AGG_AREA_UM2_COL]) * 100

print(f"\n  Column '{AGG_AREA_PERCENT_COL}' created with percentage values, relative to anatomical region area.")

# 4. Calculate the normalized intensity
# Initialize the new normalized intensity column with 0.0.
sum_intensity_per_region_with_anatomical_area[CALC_NORM_INTENSITY_COL] = 0.0
# Reuse the valid_area_mask as the same conditions apply for this normalization.
# Apply the normalized intensity calculation only to rows with valid anatomical areas.
sum_intensity_per_region_with_anatomical_area.loc[valid_area_mask, CALC_NORM_INTENSITY_COL] = \
    sum_intensity_per_region_with_anatomical_area[AGG_SUM_INTENSITY_COL] / \
    sum_intensity_per_region_with_anatomical_area[AGG_AREA_UM2_COL]

# 5. Update the main DataFrame to include all newly calculated metrics.
sum_intensity_per_region = sum_intensity_per_region_with_anatomical_area
sum_intensity_per_region.head()

## Declarative Export from CSV



This section allows you to export aggregated data for a predefined list of brain regions specified in a CSV file. 

**Instructions:**

1.  Create a CSV file with the columns: `Region Name`, `Acronym`, and `Region ID`.
2.  Place this file in the same directory as your notebook or provide a full path to it in the configuration cell below.
3.  Run the cells to generate an aggregated export file. The output will be saved in the `Exports` directory.

### Configuration

In [ ]:
# --- Configuration for Declarative Export ---
# The path to your input CSV file containing the list of regions to export.
# This file must contain the columns 'Region Name', 'Acronym', and 'Region ID'.
DECLARATIVE_INPUT_CSV_PATH = "highlighted_regions_list.csv"

# The prefix for the exported filename. The final name will be in the format:
# {PREFIX}_{YYYYMMDD-HHMMSS}.csv
DECLARATIVE_EXPORT_FILENAME_PREFIX = f"{DETECTIONS_BASE_NAME}_Declarative"

### Execution

In [ ]:
import pandas as pd
import os
import time
from pathlib import Path

def aggregate_measurements(
    region_id: int,
    group_name: str,
    measurements_df: pd.DataFrame,
    final_output_cols: list,
    selected_region_acronym: str,
) -> pd.DataFrame:
    """
    Uses the single precomputed row for this region (Model A).
    QuPath already attributed detections to every ancestor, so do not sum descendants.
    """
    df_filtered = measurements_df[measurements_df['id'] == region_id].copy()
    if df_filtered.empty:
        return pd.DataFrame()

    # One row expected; sum is safe if duplicates ever appear
    value_cols = [
        'Total Aggregated Puncta Area',
        'Total Sum Intensity',
        'Anatomical Region Area µm^2',
        'Num Estimated Total Spots',
    ]
    df_agg = pd.DataFrame(df_filtered[value_cols].sum()).T
    df_agg['Region'] = group_name
    df_agg['Acronym'] = selected_region_acronym
    df_agg['id'] = region_id

    total_area = df_agg['Anatomical Region Area µm^2'].iloc[0]
    df_agg['Total Puncta Area (%)'] = (
        (df_agg['Total Aggregated Puncta Area'] / total_area) * 100 if total_area > 0 else 0
    )
    df_agg['Normalized Puncta Intensity (per µm^2)'] = (
        df_agg['Total Sum Intensity'] / total_area if total_area > 0 else 0
    )

    for col in final_output_cols:
        if col not in df_agg.columns:
            df_agg[col] = 0
    return df_agg[final_output_cols]


def create_zero_measurement_row(
    region_id: int,
    region_name: str,
    region_acronym: str,
    annotations_area_df: pd.DataFrame,
    final_output_cols: list,
) -> pd.DataFrame:
    """
    Row for a region with no measurement data.
    Anatomical area comes only from this region, not from descendants.
    """
    data = {col: 0 for col in final_output_cols}
    data.update({
        'Region': region_name,
        'Acronym': region_acronym,
        'id': region_id,
    })

    if 'id' in annotations_area_df.columns:
        area_rows = annotations_area_df[annotations_area_df['id'] == region_id]
        data['Anatomical Region Area µm^2'] = area_rows['Anatomical Region Area µm^2'].sum()

    return pd.DataFrame([data])[final_output_cols]


def run_declarative_export(
    input_csv_path: str,
    export_prefix: str,
    export_dir: str,
    measurements_df: pd.DataFrame,
    annotations_area_df: pd.DataFrame
) -> None:
    """
    Executes the declarative export process.
    """
    print("--- Starting Declarative Export ---")

    # 1. --- Load and Validate Input CSV ---
    try:
        input_path = Path(input_csv_path)
        df_input = pd.read_csv(input_path)
        print(f"  Successfully loaded input file: '{input_path.name}'")
        required_cols = {'Region ID', 'Region Name', 'Acronym'}
        if not required_cols.issubset(df_input.columns):
            print(f"❌ Error: Input CSV is missing required columns: {required_cols - set(df_input.columns)}")
            return
    except FileNotFoundError:
        print(f"❌ Error: Input file not found at '{Path(input_csv_path).resolve()}'")
        return
    except Exception as e:
        print(f"❌ Error: Failed to load or parse input CSV '{input_csv_path}': {e}")
        return

    # 2. --- Define Final Output Schema and Aggregate Data ---
    all_aggregated_dfs = []
    # This list now defines the exact columns and order for the final CSV
    desired_output_cols = [
        'Region', 'Acronym', 'id',
        'Total Aggregated Puncta Area', 'Total Sum Intensity',
        'Num Estimated Total Spots',
        'Anatomical Region Area µm^2', 'Total Puncta Area (%)',
        'Normalized Puncta Intensity (per µm^2)',
    ]

    print(f"\n  Processing {len(df_input)} regions from the input file...")
    for _, row in df_input.iterrows():
        region_id = int(row['Region ID'])
        region_name = row['Region Name']
        region_acronym = row['Acronym']

        has_data = (measurements_df['id'] == region_id).any()

        if has_data:
            agg_df = aggregate_measurements(
                region_id,
                region_name,
                measurements_df,
                desired_output_cols,
                region_acronym,
            )
        else:
            print(f"  ℹ️ Info: No measurement data found for '{region_name}'. Creating zero-entry.")
            agg_df = create_zero_measurement_row(
                region_id,
                region_name,
                region_acronym,
                annotations_area_df,
                desired_output_cols,
            )
        all_aggregated_dfs.append(agg_df)

    # 3. --- Consolidate and Export Final DataFrame ---
    if not all_aggregated_dfs:
        print("\n❌ Error: No data was aggregated. Aborting export.")
        return

    final_df = pd.concat(all_aggregated_dfs, ignore_index=True)
    export_path = Path(export_dir)
    export_path.mkdir(parents=True, exist_ok=True)
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    filename = f"{export_prefix}_{timestamp}.csv"
    full_path = export_path / filename

    try:
        if 'id' in final_df.columns:
            final_df['id'] = final_df['id'].astype('Int64')
        final_df.to_csv(full_path, index=False)
        print(f"\n✅ Success! Data for {len(final_df)} region(s) saved to:")
        print(f"   '{full_path.resolve()}'")
    except Exception as e:
        print(f"\n❌ Error: Failed to save the final CSV file: {e}")

# --- Run the process with the configured parameters ---
if 'id' not in sum_intensity_per_region.columns:
    name_to_id_map = df_hierarchy.reset_index().set_index('name')['id'].to_dict()
    sum_intensity_per_region['id'] = sum_intensity_per_region['Region'].map(name_to_id_map)

run_declarative_export(
    input_csv_path=DECLARATIVE_INPUT_CSV_PATH,
    export_prefix=DECLARATIVE_EXPORT_FILENAME_PREFIX,
    export_dir=EXPORT_DIRECTORY,
    measurements_df=sum_intensity_per_region,
    annotations_area_df=anatomical_area_per_region
)